# Model comparison and research limits

## Objective and methodology

Compare models on the same derived held-out view after validation-only model selection. Display actual aggregate metrics, resource usage and aspect coverage. No private review/error text is displayed.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path.cwd() if (Path.cwd()/"configs/phase3_config.yaml").exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT)); sys.path.insert(0,str(ROOT/"scripts"))
result = json.loads((ROOT/"reports/phase3_results.json").read_text())
assert result["mode"] == "full", "A smoke run is not a completed experiment"
print("Dataset sizes:", {k:v["reviews"] for k,v in result["dataset"]["splits"].items()})
print("All evaluation labels are rating-derived weak labels, not verified human sentiment.")

Dataset sizes: {'train': 18367, 'validation': 2489, 'test': 2451}
All evaluation labels are rating-derived weak labels, not verified human sentiment.


## Completed run and checkpoint validation

In [2]:
from src.models.aspects.rules import aspects
display(pd.DataFrame({name:{"validation_macro_f1":r["validation"]["macro_f1"],"weak_test_accuracy":r["test"]["accuracy"],"weak_test_macro_f1":r["test"]["macro_f1"],"training_seconds":r["training_seconds"],"artifact_bytes":r["artifact_bytes"]} for name,r in result["models"].items()}).T)
print("Validation-selected model:",result["selection"]["best_feasible_model"])
print("Aspect coverage (not quality):",result["aspects"]["reviews_with_aspect_mentions"])
illustration = aspects("The camera is excellent, but the battery drains quickly.","smartphones")
print("Synthetic rule demonstration, not real annotated ground truth:",[(r["aspect"],r["sentiment"]) for r in illustration])

,validation_macro_f1,weak_test_accuracy,weak_test_macro_f1,training_seconds,artifact_bytes
logistic_regression,0.659306,0.846185,0.672835,28.919419,3028276.0
linear_svm,0.648824,0.879233,0.666704,24.320915,3028160.0
compact_bert,0.654685,0.869033,0.698821,116.715683,18262709.0


Validation-selected model: logistic_regression
Aspect coverage (not quality): 1094
Synthetic rule demonstration, not real annotated ground truth: [('camera', 'positive'), ('battery', 'negative')]


## Actual figures

External image links keep notebook outputs small.

In [3]:
figures = [name for name in result.get("figures",[]) if ("loss" in name if "06_" in '07_model_comparison' else "model_comparison" in name)]
for name in figures:
    display(Markdown("![Actual experimental figure](../reports/figures/"+name+")"))

![Actual experimental figure](../reports/figures/phase3_full_model_comparison.png)

## Interpretation and limitations

The entire phase remains partial without independent human sentiment and genuine aspect annotations. Product identities, geography and robust multilingual evaluation are unresolved. Models/checkpoints stay local; no recommendation engine or Phase 4 is implemented.